# Error-aware multitask DPO — Qwen2.5-0.5B

This experiment keeps BioASQ inference **answer-only** and trains one LoRA adapter with:

\[
L = L_{\mathrm{DPO}}(\mathrm{C3} > \mathrm{C1}) + 0.02\,L_{\mathrm{diagnostic}}
\]

The diagnostic loss is token-mean causal-LM cross entropy on a separate error-analysis prompt. The deliberately mistaken C1 rationale is input to critique; it is never a target.

The run starts from the original full-data answer-only SFT adapter, evaluates all 160 BioASQ dev questions at 4,096 tokens every 25 optimizer updates, and selects the checkpoint by official dev MRR.


In [ ]:
from pathlib import Path
import json
import os
import subprocess

ROOT = Path("/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining")
PYTHON = Path("/home/dinh-tuan/miniconda3/envs/bioasq/bin/python")
SCRIPT = ROOT / "cse_dpo/run_factoid_error_aware_multitask_dpo_qwen25_05b.py"
DATASET_ROOT = ROOT / "Artifacts/cse_dpo/error_aware_multitask_pairs/strict_equivalence_v2_accepted466_v1"
OUTPUT_ROOT = ROOT / "Artifacts/cse_dpo/error_aware_multitask_dpo_qwen25_05b_strict_equivalence_v2_accepted466_lambda002"
ENV = {**os.environ, "PYTHONPATH": str(ROOT)}

print("Script:", SCRIPT)
print("Dataset:", DATASET_ROOT)
print("Output:", OUTPUT_ROOT)


## 1. Prepare and validate only

This is safe to rerun. It rebuilds the frozen JSONL files and checks token lengths. It does **not** start training.


In [ ]:
subprocess.run(
    [str(PYTHON), str(SCRIPT)],
    cwd=ROOT,
    env=ENV,
    check=True,
)


## 2. Review the dataset before training


In [ ]:
summary = json.loads((DATASET_ROOT / "summary.json").read_text())
preflight = json.loads((DATASET_ROOT / "token_preflight.json").read_text())
display(summary)
display(preflight)

with (DATASET_ROOT / "dpo_stage1_concept_learning_all_pairs.jsonl").open() as handle:
    example = json.loads(next(handle))

display({
    key: example[key]
    for key in [
        "pair_id",
        "question",
        "chosen",
        "rejected",
        "auxiliary_error_group",
        "auxiliary_error_type",
        "auxiliary_prompt",
        "auxiliary_target",
    ]
})


## 3. Start the full run

Set RUN_TRAINING to True and run this cell. The launcher will not call GPT or modify annotations.


In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    subprocess.run(
        [str(PYTHON), str(SCRIPT), "--run"],
        cwd=ROOT,
        env=ENV,
        check=True,
    )
else:
    print("Training not started. Change RUN_TRAINING to True when ready.")


## 4. Inspect progress or the completed run


In [ ]:
stage_dir = OUTPUT_ROOT / "stage_1_concept_learning"
eval_history_path = stage_dir / "eval_history.json"
manifest_path = stage_dir / "manifest.json"

if eval_history_path.exists():
    history = json.loads(eval_history_path.read_text())
    display([
        {
            "step": row.get("step"),
            "dev_mrr": row.get("dev_mrr"),
            "pairwise_accuracy": row.get("pairwise_accuracy"),
            "dpo_eval_loss": row.get("dpo_eval_loss"),
            "auxiliary_eval_loss": row.get("auxiliary_diagnostic_eval_loss"),
            "joint_eval_loss": row.get("objective_eval_loss"),
            "retention_rate": row.get("retention_rate"),
            "newly_correct": row.get("newly_correct_count"),
            "lost_correct": row.get("lost_correct_count"),
        }
        for row in history
    ])
else:
    print("No evaluation history yet:", eval_history_path)

if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    display({
        "best_dev_mrr": manifest.get("best_dev_mrr"),
        "best_selected_adapter": manifest.get("best_selected_adapter"),
        "stopped_early": manifest.get("stopped_early"),
    })
